In [ ]:
from IPython.display import HTML, display
from collections import defaultdict


### Metric mapping

In [ ]:
source_metrics = {
    "GitHub": [
        "default_branch_protected",
        "star_count",
        "fork_count",
        "num_open_issues",
        "has_workflow_support",
        "has_distribution_support",
        "has_security_policy",
        "has_releases",
        "days_since_last_commit",
        "avg_issue_close_time_days",
        "inverse_simpson_index",
        "gitignore_present",
        "security_updates",
        "secret_scanning",
    ],
    "GitLab": [
        "default_branch_protected",
        "star_count",
        "fork_count",
        "num_open_issues",
        "has_workflow_support",
        "has_distribution_support",
        "has_security_policy",
        "has_releases",
        "days_since_last_commit",
        "avg_issue_close_time_days",
        "inverse_simpson_index",
        "gitignore_present",
    ],
    "bio.tools": [
        "input_formats",
        "output_formats",
        "biotools_registry",
    ],
    "HowFAIRis": [
        "license",
        "repository",
        "registry",
        "citation",
        "checklist",
    ],
    "OpenAlex": [
        "publication_open_access",
        "fwci",
        "citation_count",
    ],
    "Altmetric": [
        "altmetric_score",
    ],
    "Lizard": [
        "nloc_per_file",
        "avg_ccn",
        "duplicate_rate",
    ],
}

### Weights

In [ ]:

dimension_weights = {
    "compatibility": {
        "input_formats": 0.25,
        "output_formats": 0.25,
        "has_workflow_support": 0.25,
        "has_distribution_support": 0.25,
    },
    "fairness": {
        "license": 0.20,
        "repository": 0.30,
        "registry": 0.20,
        "citation": 0.10,
        "checklist": 0.10,
        "publication_open_access": 0.10,
    },
    "maintainability": {
        "nloc_per_file": 0.30,
        "avg_ccn": 0.40,
        "duplicate_rate": 0.30,
    },
    "sustainability": {
        "avg_issue_close_time_days": 0.30,
        "num_open_issues": 0.30,
        "days_since_last_commit": 0.20,
        "inverse_simpson_index": 0.10,
        "has_releases": 0.10,
    },
    "security": {
        "default_branch_protected": 0.34,
        "has_security_policy": 0.33,
        "gitignore_present": 0.01,
        "security_updates": 0.01,
        "secret_scanning": 0.01,
    },
    "scientific impact": {
        "fwci": 0.30,
        "citation_count": 0.10,
        "star_count": 0.15,
        "fork_count": 0.10,
        "altmetric_score": 0.15,
    },
}

overall_weights = {
    "compatibility": 0.20,
    "fairness": 0.20,
    "maintainability": 0.20,
    "scientific impact": 0.10,
    "security": 0.10,
    "sustainability": 0.20,
}

### Colors

In [ ]:
source_colors = {
    "GitHub": "rgb(5, 39, 252)",
    "GitLab": "rgb(252, 109, 38)",
    "bio.tools": "rgb(46, 134, 171)",
    "HowFAIRis": "rgb(123, 44, 191)",
    "OpenAlex": "rgb(25, 135, 84)",
    "Altmetric": "rgb(214, 51, 132)",
    "Lizard": "rgb(240, 173, 0)",
}

dimension_colors = {
    "compatibility": "rgb(58, 134, 255)",
    "fairness": "rgb(131, 56, 236)",
    "maintainability": "rgb(255, 0, 110)",
    "sustainability": "rgb(251, 86, 7)",
    "security": "rgb(106, 153, 78)",
    "scientific impact": "rgb(17, 138, 178)",
}

### Calculate source -> dimension flow weights

In [ ]:
# the metric layer is intentionally not drawn

source_dimension_weights = defaultdict(lambda: defaultdict(float))

for source, metrics in source_metrics.items():
    for dimension, metric_weights in dimension_weights.items():
        for metric, weight in metric_weights.items():
            if metric in metrics:
                source_dimension_weights[source][dimension] += weight

### Build diagram nodes and flows

In [ ]:
SOURCE_LAYER = 0
DIMENSION_LAYER = 1
OVERALL_LAYER = 2

nodes = []
flows = []
dimension_ids = {}

# Source nodes
for order, source in enumerate(source_metrics):
    nodes.append({
        "id": f"source:{source}",
        "label": source,
        "layer": SOURCE_LAYER,
        "order": order,
        "height": 60,
        "color": source_colors[source],
        "kind": "source",
    })


# Dimension nodes
for order, dimension in enumerate(dimension_weights):
    dimension_id = f"dimension:{dimension}"
    dimension_ids[dimension] = dimension_id

    nodes.append({
        "id": dimension_id,
        "label": dimension,
        "layer": DIMENSION_LAYER,
        "order": order,
        "height": 60,
        "color": dimension_colors[dimension],
        "kind": "dimension",
    })


# Overall-score node
nodes.append({
    "id": "overall",
    "label": "overall score",
    "layer": OVERALL_LAYER,
    "order": 0,
    "height": 150,
    "color": "rgb(0, 0, 0)",
    "kind": "overall",
})

# Source -> dimension flows
for source, contributions in source_dimension_weights.items():
    for dimension, contribution in contributions.items():
        if contribution <= 0:
            continue

        flows.append({
            "source": f"source:{source}",
            "target": dimension_ids[dimension],
            "value": contribution,
            "label": (
                f"{source} contributes "
                f"{contribution:.0%} to {dimension}"
            ),
            "kind": "source_dimension",
        })


# Dimension -> overall-score flows
for dimension, contribution in overall_weights.items():
    flows.append({
        "source": dimension_ids[dimension],
        "target": "overall",
        "value": contribution,
        "label": (
            f"{dimension} contributes "
            f"{contribution:.0%} to overall score"
        ),
        "kind": "dimension_overall",
    })


### Renderer

In [ ]:
html = r"""
<div style="
        width: 100%;
        overflow-x: auto;
        overflow-y: hidden;
        padding: 0;
        box-sizing: border-box;
    ">
    <div id="alluvial" style="width:1600px;height:750px;"></div>
</div>

<script src="https://cdn.jsdelivr.net/npm/echarts@6/dist/echarts.min.js"></script>

<script>
const nodes = __NODES__;
const flows = __FLOWS__;

const chart = echarts.init(document.getElementById('alluvial'));


// Layout

const W = 1600;
const H = 750;

const nodeWidth = 28;
const layerX = [260, 880, 1300];

const layerGapY = 18;
const topMargin = 45;
const bottomMargin = 35;


// Helpers

function groupBy(items, keyFunction) {
    const groups = new Map();

    for (const item of items) {
        const key = keyFunction(item);

        if (!groups.has(key)) {
            groups.set(key, []);
        }

        groups.get(key).push(item);
    }

    return groups;
}

function escapeHtml(value) {
    return String(value)
        .replaceAll('&', '&amp;')
        .replaceAll('<', '&lt;')
        .replaceAll('>', '&gt;')
        .replaceAll('"', '&quot;')
        .replaceAll("'", '&#039;');
}

function lighten(rgbString, factor = 0.16) {
    const [r, g, b] = rgbString.match(/\d+/g).map(Number);
    const lift = c => Math.round(c + (255 - c) * factor);
    return `rgb(${lift(r)}, ${lift(g)}, ${lift(b)})`;
}

function ribbonPath(x0, y0a, y0b, x1, y1a, y1b, curvature = 0.42) {
    const dx = x1 - x0;
    const c1 = x0 + dx * curvature;
    const c2 = x1 - dx * curvature;

    return [
        `M ${x0} ${y0a}`,
        `C ${c1} ${y0a}, ${c2} ${y1a}, ${x1} ${y1a}`,
        `L ${x1} ${y1b}`,
        `C ${c2} ${y1b}, ${c1} ${y0b}, ${x0} ${y0b}`,
        'Z'
    ].join(' ');
}


// Node placement

const nodesByLayer = groupBy(nodes, node => node.layer);
const nodeMap = new Map(nodes.map(node => [node.id, node]));

for (const [layer, layerNodes] of nodesByLayer.entries()) {
    layerNodes.sort((a, b) => a.order - b.order);

    const totalNodeHeight = layerNodes.reduce(
        (sum, node) => sum + node.height,
        0
    );

    const availableHeight = H - topMargin - bottomMargin;

    const effectiveGap = Math.max(
        layerGapY,
        (availableHeight - totalNodeHeight)
            / Math.max(1, layerNodes.length - 1)
    );

    const occupiedHeight = totalNodeHeight
        + effectiveGap * Math.max(0, layerNodes.length - 1);

    let currentY = (H - occupiedHeight) / 2;

    for (const node of layerNodes) {
        node.x = layerX[node.layer];
        node.y = currentY;

        currentY += node.height + effectiveGap;
    }
}


// Flow indexing

const outgoing = new Map(nodes.map(node => [node.id, []]));
const incoming = new Map(nodes.map(node => [node.id, []]));

for (const flow of flows) {
    outgoing.get(flow.source).push(flow);
    incoming.get(flow.target).push(flow);
}


// Allocate ribbon segments on each node face

for (const node of nodes) {
    const nodeFlows = outgoing.get(node.id);
    const totalValue = nodeFlows.reduce(
        (sum, flow) => sum + flow.value,
        0
    );

    let cursor = node.y;

    for (const flow of nodeFlows) {
        const segmentHeight = totalValue > 0
            ? (flow.value / totalValue) * node.height
            : 0;

        flow.sy0 = cursor;
        flow.sy1 = cursor + segmentHeight;

        cursor += segmentHeight;
    }
}

for (const node of nodes) {
    const nodeFlows = incoming.get(node.id);
    const totalValue = nodeFlows.reduce(
        (sum, flow) => sum + flow.value,
        0
    );

    let cursor = node.y;

    for (const flow of nodeFlows) {
        const segmentHeight = totalValue > 0
            ? (flow.value / totalValue) * node.height
            : 0;

        flow.ty0 = cursor;
        flow.ty1 = cursor + segmentHeight;

        cursor += segmentHeight;
    }
}


// Chart

const option = {
    animation: false,

    tooltip: {
        formatter: params => {
            const info = params.info || {};

            if (info.kind === 'flow') {
                return `
                    <b>${escapeHtml(info.label)}</b><br/>
                    Flow weight: ${info.value.toFixed(3)}
                `;
            }

            if (info.kind === 'node') {
                return `
                    <b>${escapeHtml(info.label)}</b><br/>
                    Type: ${escapeHtml(info.nodeKind)}
                `;
            }

            return '';
        }
    },

    graphic: [
        {
            type: 'text',
            left: 240,
            top: 12,
            style: {
                text: 'Sources',
                font: 'bold 16px sans-serif',
                fill: '#222'
            }
        },
        {
            type: 'text',
            left: 820,
            top: 12,
            style: {
                text: 'Maturity dimensions',
                font: 'bold 16px sans-serif',
                fill: '#222'
            }
        },
        {
            type: 'text',
            left: 1300,
            top: 12,
            style: {
                text: 'Final score',
                font: 'bold 16px sans-serif',
                fill: '#222'
            }
        }
    ],

    series: [{
        type: 'custom',
        coordinateSystem: 'none',

        renderItem: function(params, api) {
            const children = [];

            // Ribbons
            for (const flow of flows) {
                const source = nodeMap.get(flow.source);
                const target = nodeMap.get(flow.target);

                const x0 = source.x + nodeWidth;
                const x1 = target.x;

                children.push({
                    type: 'path',

                    shape: {
                        pathData: ribbonPath(
                            x0,
                            flow.sy0,
                            flow.sy1,
                            x1,
                            flow.ty0,
                            flow.ty1
                        )
                    },

                    style: {
                        fill: {
                            type: 'linear',
                            x: 0,
                            y: 0,
                            x2: 1,
                            y2: 0,

                            colorStops: [
                                {
                                    offset: 0,
                                    color: source.color
                                },
                                {
                                    offset: 1,
                                    color: target.color
                                }
                            ]
                        },

                        opacity: 0.72,
                        stroke: 'rgba(0, 0, 0, 0.12)',
                        lineWidth: 0.7
                    },

                    z2: 1,

                    info: {
                        kind: 'flow',
                        label: flow.label,
                        value: flow.value
                    }
                });
            }


            // Nodes and labels
            for (const node of nodes) {
                children.push({
                    type: 'rect',

                    shape: {
                        x: node.x,
                        y: node.y,
                        width: nodeWidth,
                        height: node.height,
                        r: 4
                    },

                    style: {
                        fill: lighten(node.color),
                        stroke: node.color,
                        lineWidth: 1.5
                    },

                    z2: 3,

                    info: {
                        kind: 'node',
                        label: node.label,
                        nodeKind: node.kind
                    }
                });

                let labelX;
                let labelAlign;

                if (node.layer === 0) {
                    labelX = 240;
                    labelAlign = 'right';
                } else if (node.layer === 2) {
                    labelX = node.x + nodeWidth + 12;
                    labelAlign = 'left';
                } else {
                    labelX = 895;
                    labelAlign = 'center';
                }


                children.push({
                    type: 'text',

                    x: labelX,
                    y: node.y + node.height / 2,

                    style: {
                        text: node.label,
                        textAlign: labelAlign,
                        textVerticalAlign: 'middle',

                        // Black label text
                        font: '14px sans-serif',
                        fill: '#111111',

                        // Semi-transparent white label box
                        backgroundColor: 'rgba(255, 255, 255, 0.78)',
                        borderColor: 'rgba(0, 0, 0, 0.18)',
                        borderWidth: 1,
                        borderRadius: 4,

                        // [top, right, bottom, left]
                        padding: [4, 7, 4, 7]
                    },

                    z2: 5
                });
            }

            return {
                type: 'group',
                children: children
            };
        },

        data: [0]
    }]
};

chart.setOption(option);

window.addEventListener('resize', () => chart.resize());
</script>
"""

### Chart

In [ ]:
import json

html = html.replace("__NODES__", json.dumps(nodes))
html = html.replace("__FLOWS__", json.dumps(flows))

In [ ]:
# Display chart)
display(HTML(html))

In [ ]:
# Save chart to file sankey_chart.html
from pathlib import Path

out = Path("sankey_chart.html")
out.write_text(html, encoding="utf-8")

In [ ]:
# Open chart in browser
import webbrowser

webbrowser.open(out.resolve().as_uri())